In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.append(str(project_root))

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms
import pytorch_lightning as pl
from torch.utils.data import Dataset, Subset, DataLoader

from hiermoe.constants import *
from hiermoe.extra_functions import set_seed
from hiermoe.data_setup import ImageDataset, HierImageDataset
from hiermoe.hierarchy import Hierarchy

/Users/alexandermichaeltjhin/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/alexandermichaeltjhin/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
set_seed(SEED)

DataPath = '/Users/alexandermichaeltjhin/Everything/Repos/Zooplankton/data/WHOI-Plankton'
DataSubdirectory = [
    '2006','2007','2008','2009','2010','2011', '2012',
    '2013','2014']
adj_graph = whoi_adjacency_graph_s
RESOLUTION = 65

SELECTED_LIST = [key for key, item in adj_graph.items() if len(item) == 0]

dataset = ImageDataset(
    data_directory = '/Users/alexandermichaeltjhin/Everything/Repos/Zooplankton/data/WHOI-Plankton',
    data_subdirectories = DataSubdirectory,
    class_names = SELECTED_LIST,
    max_class_size = 10000,
    image_resolution = 64,
    image_transforms = None,
    format_file = '.png',
    seed = SEED
    )

whoi_dataset = HierImageDataset(
    base_dataset=dataset,
    adjacency_graph = whoi_adjacency_graph_s,
    levels=3,
    leaves_only=True
)
whoi_dataset.print_dataset_details()

[leaves_only] Kept 70843 samples | Removed 0 non-leaf samples

Total Dataset: Size = 70843 | Levels = 3
all_node_counts: {0: 70843, 1: 54338, 3: 12148, 7: 10000, 8: 295, 9: 1853, 4: 42190, 10: 0, 11: 10000, 12: 2191, 13: 10000, 14: 0, 15: 10000, 16: 0, 17: 9999, 2: 16505, 5: 5161, 18: 0, 19: 5161, 6: 11344, 20: 10000, 21: 711, 22: 633}

nodes_by_level: {0: [0], 1: [1, 2], 2: [3, 4, 5, 6], 3: [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]}


------------------------Level 0------------------------
Level: 0 | Class Name: root                 | Class Label:   0 | Type: Parent | Count:  70843 | Prop: 1.00

------------------------Level 1------------------------
Level: 1 | Class Name: Colonial             | Class Label:   1 | Type: Parent | Count:  54338 | Prop: 0.77
Level: 1 | Class Name: Unicellular          | Class Label:   2 | Type: Parent | Count:  16505 | Prop: 0.23

------------------------Level 2------------------------
Level: 2 | Class Name: C-Spines             | Cla

In [7]:
from torch.utils.data import Dataset, Subset, DataLoader, SequentialSampler, WeightedRandomSampler, random_split
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(180),
    transforms.Pad(padding = 5, fill = 0),
    transforms.Resize((RESOLUTION, RESOLUTION)),
    transforms.ToTensor(),
])


whoi_dataset.append_image_transforms(
    image_transforms = train_transforms, replace = True
)

TRAIN_PROP = 0.7
VAL_PROP = 0.1
TEST_PROP = 0.2

BATCH_SIZE = 64

train_split, val_split, test_split = whoi_dataset.split_train_test_val(
    train_prop = TRAIN_PROP, val_prop = VAL_PROP, test_prop = TEST_PROP
)

# Create dataloaders
train_loader, val_loader, test_loader = whoi_dataset.create_dataloaders(
    batch_size = BATCH_SIZE,
    train_indices = train_split,
    val_indices = val_split,
    test_indices = test_split,
    image_transforms = None,
    train_sample_weights = None
)


In [ ]:
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print(f'Using device: MPS (Apple Silicon GPU)')
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'Using device: CUDA GPU')
else:
    device = torch.device('cpu')
    print(f'Using device: CPU')

model = HierMoeNet(whoi_dataset.hierarchy, whoi_dataset.label_to_ids)

Using device: MPS (Apple Silicon GPU)


In [ ]:
import time
from datetime import date



today = date.today()
HYPERPARAMETERS = {
    'lr': 2e-4, 
    'epochs': 40, 
    'scheduler':{'type': 'CosineAnnealingLR', 'T_max': 50},
    'early_stopping': {'patience': 3, 'delta': 0.001},
}
start = time.time()
trainer = Trainer(learning_rate=HYPERPARAMETERS['lr'], max_epochs=HYPERPARAMETERS['epochs'], device=device,
                  print_every=1, model_dir=f"efficient_net_whoi_{today}")
trainer.fit(model, train_loader, val_loader)
print(f"Total time taken {round((time.time() - start) / 60, 2)} minutes")

Training at device: mps
Directory 'efficient_net_whoi_2026-03-11' already exists — saving to 'efficient_net_whoi_2026-03-11_1' instead.
beginning training
Epoch [1/40] | Train Loss: 0.0373 | Train Acc: 0.8826 | Train F1: 0.8532 | Valid Loss: 0.0346 | Valid Acc: 0.8944 | Valid F1: 0.8659 | Saved best model (val_acc=0.8944)
Epoch [2/40] | Train Loss: 0.0365 | Train Acc: 0.8862 | Train F1: 0.8594 | Valid Loss: 0.0362 | Valid Acc: 0.8865 | Valid F1: 0.8643
Epoch [3/40] | Train Loss: 0.0358 | Train Acc: 0.8880 | Train F1: 0.8649 | Valid Loss: 0.0356 | Valid Acc: 0.8909 | Valid F1: 0.8690
Epoch [4/40] | Train Loss: 0.0353 | Train Acc: 0.8906 | Train F1: 0.8671 | Valid Loss: 0.0349 | Valid Acc: 0.8907 | Valid F1: 0.8679
Epoch [5/40] | Train Loss: 0.0351 | Train Acc: 0.8910 | Train F1: 0.8723 | Valid Loss: 0.0342 | Valid Acc: 0.8954 | Valid F1: 0.8729 | Saved best model (val_acc=0.8954)
Epoch [6/40] | Train Loss: 0.0349 | Train Acc: 0.8920 | Train F1: 0.8627 | Valid Loss: 0.0323 | Valid Acc: 0

In [18]:
model = HierMoeNet(whoi_dataset.hierarchy, whoi_dataset.label_to_ids, checkpoint_dir=trainer.model_dir)
trainer = Trainer(learning_rate=HYPERPARAMETERS['lr'], max_epochs=HYPERPARAMETERS['epochs'], device=device,
                  print_every=1, model_dir=f"efficient_net_{today}")
result = trainer.predict(model, test_loader, save=True)

Loaded checkpoint: efficient_net_whoi_2026-03-11/best_model.pt
Hierarchical Evaluation:
  Level 1: Acc=0.9704 | F1=0.9586 | Prec=0.9573 | Rec=0.9599 | n=14168
    Colonial               Acc=0.9795 | F1=0.9807 | Prec=0.9819 | Rec=0.9795 | n=10884
    Unicellular            Acc=0.9403 | F1=0.9365 | Prec=0.9326 | Rec=0.9403 | n=3284
  Level 2: Acc=0.9376 | F1=0.9244 | Prec=0.9268 | Rec=0.9222 | n=14168
    C-Spines               Acc=0.8626 | F1=0.8813 | Prec=0.9007 | Rec=0.8626 | n=2439
    C-NoSpines             Acc=0.9593 | F1=0.9550 | Prec=0.9507 | Rec=0.9593 | n=8445
    U-Spines               Acc=0.9225 | F1=0.9225 | Prec=0.9225 | Rec=0.9225 | n=1032
    U-NoSpines             Acc=0.9445 | F1=0.9389 | Prec=0.9333 | Rec=0.9445 | n=2252
  Level 3: Acc=0.8730 | F1=0.8418 | Prec=0.8429 | Rec=0.8419 | n=14168
    Chaetoceros            Acc=0.8455 | F1=0.8613 | Prec=0.8776 | Rec=0.8455 | n=2019
    Lauderia               Acc=0.5614 | F1=0.5818 | Prec=0.6038 | Rec=0.5614 | n=57
    Asterion

FileNotFoundError: [Errno 2] No such file or directory: 'efficient_net_2026-03-11/predictions.npz'

In [ ]:
from hiermoe.